# 3D reporter timelapse — 01_phase_mask_and_reporter_threshold_exploration

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 01 | Phase Mask And Reporter Threshold Exploration

This notebook sets up the first non-segmentation measurement path for the
FOXF1 / BMP4 3D organoid timelapse. The focus is organoid-level quantification:

- build a stable organoid mask from phase
- subtract technical fluorescence background using off-organoid pixels
- define reporter-positive pixels inside the organoid
- compare exploratory RFP and YFP traces over time

The goal is to decide which measurement definitions best support the manuscript
claim that RFP turns on before YFP.


## Cell Guide

1. Load the acquisition manifest and resolve the raw dataset.
2. Run a sampled scan across all positions to rank dynamic reporter candidates.
3. Review phase-mask candidates on a curated subset of positions.
4. Build early-baseline and frame-adaptive reporter thresholds.
5. Save exploratory per-position traces and preview overlays.


In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import display
from matplotlib.ticker import FuncFormatter
from scipy import ndimage as ndi
from skimage import filters, measure, morphology, segmentation

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.rcParams["figure.dpi"] = 120


In [ ]:
# -------------------------------
# User configuration
# -------------------------------
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "scripts").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

POSITION_MANIFEST_PATH = ROOT / "results/manifests/acquisition_position_manifest.tsv"
position_manifest = pd.read_csv(POSITION_MANIFEST_PATH, sep="\t")

dataset_rel = position_manifest["dataset_dir"].iloc[0]
DATASET_DIR = ROOT / dataset_rel
POSITION_RE = re.compile(r"Pos(?P<index>\d+)$")
CHANNEL_LABELS = {1: "RFP", 2: "YFP"}
INTERVAL_HOURS = float(position_manifest["interval_ms"].dropna().iloc[0]) / 3_600_000.0
TIME_DISPLAY_OFFSET_HOURS = 48.0

COARSE_SCAN_PATH = ROOT / "results/qc/01_coarse_reporter_scan_by_position_sampled.tsv"
COARSE_SUMMARY_PATH = ROOT / "results/qc/01_coarse_reporter_position_summary.tsv"
MASK_SUMMARY_PATH = ROOT / "results/tables/01_phase_mask_candidate_summary.tsv"
THRESHOLD_SUMMARY_PATH = ROOT / "results/tables/01_subset_threshold_summary.tsv"
METRICS_PATH = ROOT / "results/tables/01_subset_reporter_metrics.tsv"
PREVIEW_DIR = ROOT / "results/previews/01_phase_mask_and_reporter_threshold_exploration"
FIGURE_DIR = ROOT / "results/figures/01"
TRACE_FIGURE_PATH = FIGURE_DIR / "01_subset_reporter_traces.png"
POPULATION_TRACE_PATH = FIGURE_DIR / "01_subset_population_reporter_traces.png"

SAMPLED_SCAN_STEP = 25
AUTO_POSITIVE_COUNT = 6
AUTO_NEGATIVE_COUNT = 2
CURATED_POSITIONS = None

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PHASE_MASK_METHODS = ["dark_otsu", "residual_dark", "edge_fill"]
DEFAULT_MASK_METHOD = "residual_dark"
LOWER_TAIL_FRACTION = 0.60
THRESHOLD_Z_VALUES = [3.0, 4.0, 5.0]
DEFAULT_THRESHOLD_Z = 4.0
THRESHOLD_MODES = ["early_baseline", "frame_lower_tail"]
DEFAULT_THRESHOLD_MODE = "early_baseline"
BACKGROUND_RING_INNER = 4
BACKGROUND_RING_OUTER = 12

WRITE_OUTPUTS = True
SAVE_FIGURES = True

observed_frame_max = int(position_manifest["frame_index_max"].max())
max_time = observed_frame_max
MASK_REVIEW_FRAMES = sorted({0, max_time // 2, max_time})
OVERVIEW_FRAMES = sorted({0, max_time // 3, (2 * max_time) // 3, max_time})
EARLY_BASELINE_FRAMES = list(range(0, min(50, max_time + 1), 5))

for path in [
    COARSE_SCAN_PATH,
    COARSE_SUMMARY_PATH,
    MASK_SUMMARY_PATH,
    THRESHOLD_SUMMARY_PATH,
    METRICS_PATH,
    TRACE_FIGURE_PATH,
    POPULATION_TRACE_PATH,
]:
    path.parent.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

import sys
qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    display_time_hours_from_index as _display_time_hours_from_index,
    set_display_time_axis as _set_display_time_axis,
)

print("Project root:", ROOT)
print("Dataset dir:", DATASET_DIR)
print("Observed max frame:", max_time)
print("Mask review frames:", MASK_REVIEW_FRAMES)
print("Overview frames:", OVERVIEW_FRAMES)


## Output Files

This notebook writes:

- sampled whole-dataset reporter scan tables in `results/qc/`
- phase-mask candidate summary in `results/tables/`
- threshold summary and exploratory traces in `results/tables/`
- mask and reporter overlay previews in `results/previews/01_phase_mask_and_reporter_threshold_exploration/`
- exploratory trace figures in `results/figures/`


In [ ]:
# -------------------------------
# Helper functions
# -------------------------------
def pos_index_from_label(label: str) -> int:
    match = POSITION_RE.fullmatch(label)
    if not match:
        raise ValueError(f"Unexpected position label: {label}")
    return int(match.group("index"))


def display_time_hours_from_index(time_index: int | float) -> float:
    return _display_time_hours_from_index(
        time_index,
        interval_hours=INTERVAL_HOURS,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return f"{display_time_hours_from_index(time_index):.{decimals}f} h"


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(
        ax,
        axis=axis,
        crowded=crowded,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    if "time_index" in output.columns:
        output["time_hours"] = output["time_index"].map(display_time_hours_from_index)
    return output


def frame_path(position_label: str, channel_index: int, time_index: int) -> Path:
    pos_index = pos_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel{channel_index:03d}_position{pos_index:03d}_time{time_index:09d}_z000.tif"
    )


def load_frame(position_label: str, channel_index: int, time_index: int) -> np.ndarray:
    return tiff.imread(frame_path(position_label, channel_index, time_index))


def center_component(mask: np.ndarray, min_size: int = 200) -> np.ndarray:
    mask = morphology.remove_small_objects(mask.astype(bool), min_size=min_size)
    mask = ndi.binary_fill_holes(mask)
    labels = measure.label(mask)
    if labels.max() == 0:
        return mask.astype(bool)

    center = np.array(mask.shape) / 2.0
    best_label = None
    best_score = None
    for region in measure.regionprops(labels):
        area = float(region.area)
        centroid = np.array(region.centroid)
        distance = float(np.linalg.norm(centroid - center))
        score = distance - 0.002 * area
        if best_score is None or score < best_score:
            best_score = score
            best_label = region.label

    out = labels == best_label
    out = morphology.binary_closing(out, morphology.disk(5))
    out = ndi.binary_fill_holes(out)
    return out.astype(bool)


def phase_mask_candidates(phase_image: np.ndarray) -> dict[str, np.ndarray]:
    phase = phase_image.astype(float)
    blur = filters.gaussian(phase, sigma=1.5, preserve_range=True)
    coarse = filters.gaussian(blur, sigma=12, preserve_range=True)
    residual_dark = coarse - blur
    edge = filters.sobel(blur)

    dark_raw = blur < filters.threshold_otsu(blur)
    residual_raw = residual_dark > filters.threshold_otsu(residual_dark)
    edge_raw = edge > filters.threshold_otsu(edge)
    edge_raw = morphology.binary_closing(edge_raw, morphology.disk(4))
    edge_raw = ndi.binary_fill_holes(edge_raw)

    return {
        "dark_otsu": center_component(dark_raw),
        "residual_dark": center_component(residual_raw),
        "edge_fill": center_component(edge_raw),
    }


def build_phase_mask(phase_image: np.ndarray, method: str = DEFAULT_MASK_METHOD) -> np.ndarray:
    candidates = phase_mask_candidates(phase_image)
    if method not in candidates:
        raise KeyError(f"Unknown phase mask method: {method}")
    return candidates[method]


def annulus_mask(organoid_mask: np.ndarray, inner: int = BACKGROUND_RING_INNER, outer: int = BACKGROUND_RING_OUTER) -> np.ndarray:
    inner_mask = morphology.binary_dilation(organoid_mask, morphology.disk(inner))
    outer_mask = morphology.binary_dilation(organoid_mask, morphology.disk(outer))
    ring = outer_mask & ~inner_mask
    if ring.sum() < 100:
        ring = ~outer_mask
    return ring


def robust_threshold(values: np.ndarray, fraction: float = LOWER_TAIL_FRACTION, z: float = DEFAULT_THRESHOLD_Z) -> tuple[float, float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan"), float("nan"), float("nan")
    cutoff = np.quantile(arr, fraction)
    baseline = arr[arr <= cutoff]
    location = float(np.median(baseline))
    mad = float(np.median(np.abs(baseline - location)))
    scale = max(1.4826 * mad, 1.0)
    threshold = location + z * scale
    return float(threshold), location, float(scale)


def corrected_signal_and_mask(position_label: str, time_index: int, channel_index: int, mask_method: str = DEFAULT_MASK_METHOD) -> dict:
    phase = load_frame(position_label, 0, time_index)
    organoid_mask = build_phase_mask(phase, method=mask_method)
    ring = annulus_mask(organoid_mask)
    signal = load_frame(position_label, channel_index, time_index).astype(float)
    background_value = float(np.median(signal[ring])) if np.any(ring) else float(np.median(signal[~organoid_mask]))
    corrected = signal - background_value
    return {
        "phase": phase,
        "organoid_mask": organoid_mask,
        "ring_mask": ring,
        "signal": signal,
        "corrected": corrected,
        "background_value": background_value,
    }


def display_image(image: np.ndarray, low_q: float = 1.0, high_q: float = 99.0) -> np.ndarray:
    low, high = np.percentile(image, [low_q, high_q])
    if math.isclose(high, low):
        high = low + 1.0
    scaled = np.clip((image - low) / (high - low), 0, 1)
    return scaled


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    return segmentation.find_boundaries(mask.astype(bool), mode="outer")


def draw_mask_contour(ax, mask: np.ndarray, color: str = "cyan", linewidth: float = 1.8) -> None:
    if np.any(mask):
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth)


In [ ]:
# -------------------------------
# Sampled whole-dataset scan
# -------------------------------
positions = sorted(position_manifest["position_label"].tolist(), key=pos_index_from_label)
sampled_times = list(range(0, max_time + 1, SAMPLED_SCAN_STEP))
if sampled_times[-1] != max_time:
    sampled_times.append(max_time)

coarse_rows = []
for position_label in positions:
    for time_index in sampled_times:
        row = {"position_label": position_label, "position_index": pos_index_from_label(position_label), "time_index": time_index}
        for channel_index, reporter in CHANNEL_LABELS.items():
            image = load_frame(position_label, channel_index, time_index).astype(float)
            row[f"{reporter.lower()}_mean"] = float(image.mean())
            row[f"{reporter.lower()}_p99"] = float(np.percentile(image, 99))
            row[f"{reporter.lower()}_max"] = float(image.max())
        coarse_rows.append(row)

coarse_scan_df = pd.DataFrame(coarse_rows).sort_values(["position_index", "time_index"]).reset_index(drop=True)

summary_rows = []
for position_label, group in coarse_scan_df.groupby("position_label", sort=False):
    group = group.sort_values("time_index")
    record = {"position_label": position_label, "position_index": int(group["position_index"].iloc[0])}
    combined_range = 0.0
    for reporter in ["rfp", "yfp"]:
        record[f"{reporter}_mean_range"] = float(group[f"{reporter}_mean"].max() - group[f"{reporter}_mean"].min())
        record[f"{reporter}_p99_range"] = float(group[f"{reporter}_p99"].max() - group[f"{reporter}_p99"].min())
        record[f"{reporter}_late_minus_early_p99"] = float(group[f"{reporter}_p99"].iloc[-1] - group[f"{reporter}_p99"].iloc[0])
        combined_range += record[f"{reporter}_p99_range"]
    record["combined_p99_range"] = combined_range
    summary_rows.append(record)

coarse_summary_df = pd.DataFrame(summary_rows).sort_values("combined_p99_range", ascending=False).reset_index(drop=True)

if CURATED_POSITIONS is None:
    positive_seed = coarse_summary_df.head(AUTO_POSITIVE_COUNT)["position_label"].tolist()
    negative_seed = coarse_summary_df.tail(AUTO_NEGATIVE_COUNT)["position_label"].tolist()
    CURATED_POSITIONS = positive_seed + negative_seed

curated_set = set(CURATED_POSITIONS)
curated_summary_df = coarse_summary_df.loc[coarse_summary_df["position_label"].isin(curated_set)].copy()
curated_summary_df["curation_group"] = np.where(
    curated_summary_df["position_label"].isin(coarse_summary_df.head(AUTO_POSITIVE_COUNT)["position_label"]),
    "dynamic_seed",
    "low_dynamic_seed",
)

if WRITE_OUTPUTS:
    coarse_scan_df.to_csv(COARSE_SCAN_PATH, sep="\t", index=False)
    coarse_summary_df.to_csv(COARSE_SUMMARY_PATH, sep="\t", index=False)

print("Curated positions:", CURATED_POSITIONS)
print("\nTop dynamic positions")
display(display_time_df(coarse_summary_df.head(10)))
print("\nLow dynamic positions")
display(display_time_df(coarse_summary_df.tail(10).sort_values("combined_p99_range")))


In [ ]:
# -------------------------------
# Phase-mask candidate review
# -------------------------------
mask_rows = []
for position_label in CURATED_POSITIONS:
    fig, axes = plt.subplots(len(MASK_REVIEW_FRAMES), len(PHASE_MASK_METHODS), figsize=(4 * len(PHASE_MASK_METHODS), 3.5 * len(MASK_REVIEW_FRAMES)))
    if len(MASK_REVIEW_FRAMES) == 1:
        axes = np.expand_dims(axes, axis=0)
    if len(PHASE_MASK_METHODS) == 1:
        axes = np.expand_dims(axes, axis=1)

    for row_index, time_index in enumerate(MASK_REVIEW_FRAMES):
        phase = load_frame(position_label, 0, time_index)
        candidates = phase_mask_candidates(phase)
        for col_index, method in enumerate(PHASE_MASK_METHODS):
            mask = candidates[method]
            ax = axes[row_index, col_index]
            ax.imshow(display_image(phase), cmap="gray")
            draw_mask_contour(ax, mask, color="deepskyblue", linewidth=2.2)
            ax.set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)} | {method}")
            ax.axis("off")
            mask_rows.append(
                {
                    "position_label": position_label,
                    "time_index": time_index,
                    "mask_method": method,
                    "mask_area_px": int(mask.sum()),
                    "mask_area_fraction": float(mask.mean()),
                }
            )

    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(PREVIEW_DIR / f"{position_label}_phase_mask_candidates.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

mask_summary_df = pd.DataFrame(mask_rows).sort_values(["position_label", "time_index", "mask_method"]).reset_index(drop=True)
if WRITE_OUTPUTS:
    mask_summary_df.to_csv(MASK_SUMMARY_PATH, sep="\t", index=False)

display(display_time_df(mask_summary_df))


In [ ]:
# -------------------------------
# Build baseline thresholds on curated subset
# -------------------------------
threshold_rows = []
baseline_thresholds = {}
baseline_locations = {}
baseline_scales = {}

for position_label in CURATED_POSITIONS:
    baseline_thresholds[position_label] = {}
    for channel_index, reporter in CHANNEL_LABELS.items():
        pooled_values = []
        for time_index in EARLY_BASELINE_FRAMES:
            result = corrected_signal_and_mask(position_label, time_index, channel_index, mask_method=DEFAULT_MASK_METHOD)
            pooled_values.append(result["corrected"][result["organoid_mask"]])
        pooled_values = np.concatenate(pooled_values)
        baseline_thresholds[position_label][reporter] = {}
        baseline_locations.setdefault(position_label, {})[reporter] = {}
        baseline_scales.setdefault(position_label, {})[reporter] = {}
        for z in THRESHOLD_Z_VALUES:
            threshold, location, scale = robust_threshold(pooled_values, fraction=LOWER_TAIL_FRACTION, z=z)
            baseline_thresholds[position_label][reporter][z] = threshold
            baseline_locations[position_label][reporter][z] = location
            baseline_scales[position_label][reporter][z] = scale
            threshold_rows.append(
                {
                    "position_label": position_label,
                    "reporter": reporter,
                    "threshold_mode": "early_baseline",
                    "threshold_z": z,
                    "threshold_value": threshold,
                    "baseline_location": location,
                    "baseline_scale": scale,
                    "baseline_frame_count": len(EARLY_BASELINE_FRAMES),
                    "mask_method": DEFAULT_MASK_METHOD,
                }
            )

threshold_summary_df = pd.DataFrame(threshold_rows).sort_values(["position_label", "reporter", "threshold_z"]).reset_index(drop=True)
if WRITE_OUTPUTS:
    threshold_summary_df.to_csv(THRESHOLD_SUMMARY_PATH, sep="\t", index=False)

display(display_time_df(threshold_summary_df))


In [ ]:
# -------------------------------
# Exploratory metrics over time
# -------------------------------
metric_rows = []
frame_sequence = list(range(0, max_time + 1))

for position_label in CURATED_POSITIONS:
    for time_index in frame_sequence:
        phase = load_frame(position_label, 0, time_index)
        organoid_mask = build_phase_mask(phase, method=DEFAULT_MASK_METHOD)
        organoid_area_px = int(organoid_mask.sum())

        for channel_index, reporter in CHANNEL_LABELS.items():
            result = corrected_signal_and_mask(position_label, time_index, channel_index, mask_method=DEFAULT_MASK_METHOD)
            corrected = result["corrected"]
            organoid_values = corrected[organoid_mask]

            for threshold_mode in THRESHOLD_MODES:
                for z in THRESHOLD_Z_VALUES:
                    if threshold_mode == "early_baseline":
                        threshold_value = baseline_thresholds[position_label][reporter][z]
                        baseline_location = baseline_locations[position_label][reporter][z]
                        baseline_scale = baseline_scales[position_label][reporter][z]
                    else:
                        threshold_value, baseline_location, baseline_scale = robust_threshold(
                            organoid_values,
                            fraction=LOWER_TAIL_FRACTION,
                            z=z,
                        )

                    positive_mask = organoid_mask & (corrected > threshold_value)
                    positive_values = corrected[positive_mask]
                    metric_rows.append(
                        {
                            "position_label": position_label,
                            "position_index": pos_index_from_label(position_label),
                            "time_index": time_index,
                            "reporter": reporter,
                            "threshold_mode": threshold_mode,
                            "threshold_z": z,
                            "mask_method": DEFAULT_MASK_METHOD,
                            "organoid_area_px": organoid_area_px,
                            "background_value": result["background_value"],
                            "threshold_value": float(threshold_value),
                            "baseline_location": float(baseline_location),
                            "baseline_scale": float(baseline_scale),
                            "positive_pixel_count": int(positive_mask.sum()),
                            "positive_fraction": float(positive_mask.sum() / organoid_area_px) if organoid_area_px > 0 else float("nan"),
                            "positive_mean_intensity": float(np.mean(positive_values)) if positive_values.size else float("nan"),
                            "positive_integrated_intensity": float(np.sum(positive_values)) if positive_values.size else 0.0,
                            "organoid_mean_intensity": float(np.mean(organoid_values)) if organoid_values.size else float("nan"),
                        }
                    )

metrics_df = pd.DataFrame(metric_rows).sort_values(
    ["position_index", "reporter", "threshold_mode", "threshold_z", "time_index"]
).reset_index(drop=True)

if WRITE_OUTPUTS:
    metrics_df.to_csv(METRICS_PATH, sep="\t", index=False)

display(display_time_df(metrics_df.head()))


In [ ]:
# -------------------------------
# Overlay previews with default settings
# -------------------------------
default_metrics_df = metrics_df.loc[
    (metrics_df["threshold_mode"] == DEFAULT_THRESHOLD_MODE)
    & (metrics_df["threshold_z"] == DEFAULT_THRESHOLD_Z)
].copy()

for position_label in CURATED_POSITIONS:
    fig, axes = plt.subplots(3, len(OVERVIEW_FRAMES), figsize=(4 * len(OVERVIEW_FRAMES), 9))
    if len(OVERVIEW_FRAMES) == 1:
        axes = np.expand_dims(axes, axis=1)

    thresholds = {
        reporter: float(
            threshold_summary_df.loc[
                (threshold_summary_df["position_label"] == position_label)
                & (threshold_summary_df["reporter"] == reporter)
                & (threshold_summary_df["threshold_mode"] == "early_baseline")
                & (threshold_summary_df["threshold_z"] == DEFAULT_THRESHOLD_Z),
                "threshold_value",
            ].iloc[0]
        )
        for reporter in CHANNEL_LABELS.values()
    }

    for col_index, time_index in enumerate(OVERVIEW_FRAMES):
        phase = load_frame(position_label, 0, time_index)
        organoid_mask = build_phase_mask(phase, method=DEFAULT_MASK_METHOD)

        axes[0, col_index].imshow(display_image(phase), cmap="gray")
        draw_mask_contour(axes[0, col_index], organoid_mask, color="deepskyblue", linewidth=2.2)
        axes[0, col_index].set_title(f"{position_label} t={format_display_hours_from_index(time_index, 0)} phase")
        axes[0, col_index].axis("off")

        for row_index, (channel_index, reporter) in enumerate(CHANNEL_LABELS.items(), start=1):
            result = corrected_signal_and_mask(position_label, time_index, channel_index, mask_method=DEFAULT_MASK_METHOD)
            corrected = result["corrected"]
            positive_mask = organoid_mask & (corrected > thresholds[reporter])

            ax = axes[row_index, col_index]
            ax.imshow(display_image(corrected), cmap="gray")
            draw_mask_contour(ax, organoid_mask, color="deepskyblue", linewidth=2.0)
            draw_mask_contour(ax, positive_mask, color="magenta", linewidth=1.7)
            ax.set_title(f"{reporter} corrected | thr={thresholds[reporter]:.0f}")
            ax.axis("off")

    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(PREVIEW_DIR / f"{position_label}_reporter_overlay_review.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


In [ ]:
# -------------------------------
# Exploratory trace figures
# -------------------------------
dynamic_seed_positions = coarse_summary_df.head(AUTO_POSITIVE_COUNT)["position_label"].tolist()
default_metrics_df["seed_group"] = np.where(
    default_metrics_df["position_label"].isin(dynamic_seed_positions),
    "dynamic_seed",
    "low_dynamic_seed",
)

fig, axes = plt.subplots(len(CURATED_POSITIONS), 2, figsize=(14, 3.0 * len(CURATED_POSITIONS)), sharex=True)
if len(CURATED_POSITIONS) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, position_label in enumerate(CURATED_POSITIONS):
    subset = default_metrics_df.loc[default_metrics_df["position_label"] == position_label]
    for reporter, color in [("RFP", "tab:red"), ("YFP", "goldenrod")]:
        rep = subset.loc[subset["reporter"] == reporter]
        axes[row_index, 0].plot(rep["time_index"], rep["positive_fraction"], color=color, label=reporter)
        axes[row_index, 1].plot(rep["time_index"], rep["positive_mean_intensity"], color=color, label=reporter)

    axes[row_index, 0].set_ylabel(position_label)
    axes[row_index, 0].set_title("Positive fraction")
    axes[row_index, 1].set_title("Mean positive-pixel intensity")
    if row_index == 0:
        axes[row_index, 0].legend(loc="upper left")
        axes[row_index, 1].legend(loc="upper left")

axes[-1, 0].set_xlabel("Time (hours)")
axes[-1, 1].set_xlabel("Time (hours)")
set_display_time_axis(axes[-1, 0], "x")
set_display_time_axis(axes[-1, 1], "x")
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(TRACE_FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.close(fig)

population = (
    default_metrics_df.loc[default_metrics_df["seed_group"] == "dynamic_seed"]
    .groupby(["time_index", "reporter"], as_index=False)
    .agg(
        positive_fraction_median=("positive_fraction", "median"),
        positive_fraction_q25=("positive_fraction", lambda s: float(np.quantile(s, 0.25))),
        positive_fraction_q75=("positive_fraction", lambda s: float(np.quantile(s, 0.75))),
        positive_mean_intensity_median=("positive_mean_intensity", "median"),
        positive_mean_intensity_q25=("positive_mean_intensity", lambda s: float(np.nanquantile(s, 0.25))),
        positive_mean_intensity_q75=("positive_mean_intensity", lambda s: float(np.nanquantile(s, 0.75))),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for reporter, color in [("RFP", "tab:red"), ("YFP", "goldenrod")]:
    rep = population.loc[population["reporter"] == reporter]
    axes[0].plot(rep["time_index"], rep["positive_fraction_median"], color=color, label=reporter)
    axes[0].fill_between(rep["time_index"], rep["positive_fraction_q25"], rep["positive_fraction_q75"], color=color, alpha=0.2)
    axes[1].plot(rep["time_index"], rep["positive_mean_intensity_median"], color=color, label=reporter)
    axes[1].fill_between(rep["time_index"], rep["positive_mean_intensity_q25"], rep["positive_mean_intensity_q75"], color=color, alpha=0.2)

axes[0].set_title("Dynamic-seed positions: positive fraction")
axes[1].set_title("Dynamic-seed positions: mean positive-pixel intensity")
axes[0].set_xlabel("Time (hours)")
axes[1].set_xlabel("Time (hours)")
set_display_time_axis(axes[0], "x")
set_display_time_axis(axes[1], "x")
axes[0].legend(loc="upper left")
axes[1].legend(loc="upper left")
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(POPULATION_TRACE_PATH, dpi=180, bbox_inches="tight")
plt.close(fig)


In [ ]:
created_files = [
    COARSE_SCAN_PATH,
    COARSE_SUMMARY_PATH,
    MASK_SUMMARY_PATH,
    THRESHOLD_SUMMARY_PATH,
    METRICS_PATH,
    TRACE_FIGURE_PATH,
    POPULATION_TRACE_PATH,
]

print("Created files")
for path in created_files:
    if path.exists():
        print("-", path.relative_to(ROOT).as_posix())

print("\nPreview directory")
print("-", PREVIEW_DIR.relative_to(ROOT).as_posix())
